# Trabalho Final - Parte 2 - Parcial



## Processos em Python

### GIL do Python

O GIL (Global Interpreter Lock) é um bloqueio global que garante que apenas uma thread de Python bytecode esteja executando por vez dentro de um processo.

Isso significa que, mesmo criando várias threads, apenas uma thread consegue rodar instruções Python de cada vez.

Ou seja, por padrão threads em Python não aceleram tarefas CPU-bound.

### Multithreading e Multiprocessing

É possível criar múltiplas threads em Python através do módulo `threading` dentro de um mesmo processo Python, porém segue válida a limitação de executar apenas uma thread por vez dentro do mesmo processador, ou seja, as threads realizam chaveamento entre si em um processador, sem de fato executar em paralelo.

```python
import threading
import time

def tarefa(nome):
    for i in range(3):
        print(f"Tarefa {nome}: {i}")
        time.sleep(1)

# Criando duas threads
t1 = threading.Thread(target=tarefa, args=("A",))
t2 = threading.Thread(target=tarefa, args=("B",))

t1.start()
t2.start()

t1.join()
t2.join()
```


Com o objetivo de utilizar **múltiplos núcleos de CPU**, Python possui o módulo `multiprocessing`. Esse módulo cria **processos independentes** cada um com seu próprio interpretador e sua própria memória, evitando o GIL. Como resultado várias tarefas podem efetivamente executar em paralelo.

```python
from multiprocessing import Process
import math

def calcular():
    print(sum(i*i for i in range(10_000_000)))

processos = [Process(target=calcular) for _ in range(4)]

for p in processos:
    p.start()
for p in processos:
    p.join()
```

## Afinidade de CPU (CPU affinity)

Por padrão o kernel do Linux é livre para mover um processo entre quaisquer núcleos de CPU disponíveis no sistema. No Ubuntu (Linux), entretanto, é possível controlar em qual CPU cada processo da aplicação Python irá executar com o uso da **afinidade de CPU**. 

Cada processo no Linux pode ter uma **máscara de afinidade** indicando quais núcleos ele está autorizado a executar.

Em Python, no Linux, a afinidade pode ser definidade utilizando `os.sched_setaffinity`. A chamada de sistema pode


## Cargas de trabalho CPU-bound

- Criptografia
- Multiplicação de Matrizes

# Criptografia

- AES-CTR

In [1]:
#!pip install pycryptodome

In [2]:
import sys
print(sys.executable)

/usr/bin/python3


In [3]:
import Cryptodome
print(Cryptodome.__version__)

3.20.0


In [4]:
# Parallel AES-CTR Benchmark Script (PyCryptodome)

import os
import time
from Cryptodome.Cipher import AES
from concurrent.futures import ThreadPoolExecutor

# Parameters
KEY = os.urandom(32)   # AES-256 key
NONCE = os.urandom(8)  # 64-bit nonce
THREADS = 8
BLOCK_SIZE = 16  # AES block size in bytes

def encrypt_chunk(chunk, counter_start):
    """Encrypt one chunk with AES-CTR using a thread-specific counter offset."""
    cipher = AES.new(KEY, AES.MODE_CTR, nonce=NONCE, initial_value=counter_start)
    return cipher.encrypt(chunk)

def parallel_encrypt(data, num_threads=THREADS):
    """Encrypt data in parallel with exactly num_threads threads."""
    chunk_size = len(data) // num_threads
    chunks = [data[i*chunk_size:(i+1)*chunk_size] for i in range(num_threads)]
    
    # Assign counters so keystreams don't overlap
    counters = [i * (chunk_size // BLOCK_SIZE) for i in range(num_threads)]
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        encrypted_chunks = list(executor.map(encrypt_chunk, chunks, counters))
    
    return b''.join(encrypted_chunks)

def run_benchmark(data_size_bytes):
    """Run encryption benchmark for a given data size (bytes)."""
    print(f"\n=== Benchmark: {data_size_bytes / (1024*1024)} MB ===")
    data = os.urandom(data_size_bytes)  # Allocate test data
    
    start = time.perf_counter()
    ciphertext = parallel_encrypt(data)
    end = time.perf_counter()
    
    elapsed = end - start
    throughput_mb_s = (data_size_bytes / (1024*1024)) / elapsed
    
    print(f"Time: {elapsed:.2f} s")
    print(f"Throughput: {throughput_mb_s:.2f} MB/s using {THREADS} threads")

if __name__ == "__main__":
    sizes = [
        512 * 1024 * 1024,   # 512 MB
        1024 * 1024 * 1024,  # 1 GB
        2 * 1024 * 1024 * 1024  # 2 GB
    ]
    for size in sizes:
        run_benchmark(size)


=== Benchmark: 512.0 MB ===
Time: 2.11 s
Throughput: 242.54 MB/s using 8 threads

=== Benchmark: 1024.0 MB ===
Time: 4.34 s
Throughput: 236.13 MB/s using 8 threads

=== Benchmark: 2048.0 MB ===
Time: 9.06 s
Throughput: 226.00 MB/s using 8 threads


In [5]:
# Correctness Check

from Cryptodome.Cipher import AES

def decrypt_chunk(chunk, counter_start):
    cipher = AES.new(KEY, AES.MODE_CTR, nonce=NONCE, initial_value=counter_start)
    return cipher.decrypt(chunk)

def parallel_decrypt(ciphertext, num_threads=THREADS):
    chunk_size = len(ciphertext) // num_threads
    chunks = []
    start = 0
    for i in range(num_threads):
        end = start + chunk_size
        if i == num_threads - 1:
            end = len(ciphertext)
        chunks.append(ciphertext[start:end])
        start = end

    blocks_per_chunk = (len(ciphertext) + num_threads - 1) // num_threads // BLOCK_SIZE
    counters = [i * blocks_per_chunk for i in range(num_threads)]

    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        decrypted_chunks = list(executor.map(decrypt_chunk, chunks, counters))

    return b''.join(decrypted_chunks)

data_size_bytes = 512 * 1024 * 1024 # 512 MB
data = os.urandom(data_size_bytes)
ciphertext = parallel_encrypt(data)
recovered = parallel_decrypt(ciphertext)
assert recovered == data, "Decryption failed! Data does not match."
print("Encryption verified successfully!")

Encryption verified successfully!


In [1]:
# Install only core parts of dask
!pip install "dask[complete]"

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

# Multiplicação de Matrizes

In [6]:
"""
Safe Dask Matrix Multiplication Script
--------------------------------------
- MATRIX_SIZE is configurable (default 8000)
- NUM_WORKERS sets exactly 8 parallel Dask worker processes
- THREADS_PER_WORKER = 1 for BLAS / NumPy
- Before computing, checks that MATRIX_SIZE is divisible by NUM_WORKERS
"""

import os
import numpy as np
import dask.array as da
from dask.distributed import Client
import time
import sys

# -----------------------------
# CONFIGURATION
MATRIX_SIZE = 16000       # Size of NxN matrices
NUM_WORKERS = 8          # Number of Dask worker processes
THREADS_PER_WORKER = 1   # Threads per worker (BLAS / NumPy)

# -----------------------------
# Safety check: ensure MATRIX_SIZE is divisible by NUM_WORKERS
if MATRIX_SIZE % NUM_WORKERS != 0:
    print(f"Error: MATRIX_SIZE ({MATRIX_SIZE}) is not divisible by NUM_WORKERS ({NUM_WORKERS}).")
    print("Please choose a MATRIX_SIZE divisible by NUM_WORKERS.")
    sys.exit(1)

# -----------------------------
# Set BLAS threading to 1 per worker to avoid oversubscription
os.environ["OMP_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["MKL_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREADS_PER_WORKER)

# -----------------------------
# Start Dask client
client = Client(n_workers=NUM_WORKERS, threads_per_worker=THREADS_PER_WORKER)
print(f"Dask client started with {NUM_WORKERS} workers")

# -----------------------------
# Define chunk sizes
chunk_rows = MATRIX_SIZE // NUM_WORKERS
chunk_cols = MATRIX_SIZE // NUM_WORKERS

# Create Dask arrays
A = da.random.random((MATRIX_SIZE, MATRIX_SIZE), chunks=(chunk_rows, MATRIX_SIZE))
B = da.random.random((MATRIX_SIZE, MATRIX_SIZE), chunks=(MATRIX_SIZE, chunk_cols))

# Print chunk sizes for verification
print(f"A chunks: {A.chunks}")
print(f"B chunks: {B.chunks}")

# -----------------------------
# Matrix multiplication
start_time = time.time()
C = A @ B
result = C.compute()  # Trigger parallel execution
end_time = time.time()

print("Matrix multiplication completed")
print(f"Result shape: {result.shape}")
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

# -----------------------------
# Clean up
client.close()


Dask client started with 8 workers
A chunks: ((2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000), (16000,))
B chunks: ((16000,), (2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000))


Task exception was never retrieved
future: <Task finished name='Task-79803' coro=<Client._gather.<locals>.wait() done, defined at C:\Users\joaol\AppData\Local\Programs\Python\Python313\Lib\site-packages\distributed\client.py:2385> exception=AllExit()>
Traceback (most recent call last):
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python313\Lib\site-packages\distributed\client.py", line 2394, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 